<a href="https://colab.research.google.com/github/momulaharinathreddy74-cyber/BDA_Assignment_118/blob/main/BDA_Assignment_2_118.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Build a classification model with spark with a dataset of your choice (Iris dataset)**

In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("Iris Classification") \
    .getOrCreate()

In [ ]:
import pandas as pd
from sklearn.datasets import load_iris
iris = load_iris()
df = pd.DataFrame(data=iris.data, columns=iris.feature_names)
df['label'] = iris.target
spark_df = spark.createDataFrame(df)
spark_df.show(5)

+-----------------+----------------+-----------------+----------------+-----+
|sepal length (cm)|sepal width (cm)|petal length (cm)|petal width (cm)|label|
+-----------------+----------------+-----------------+----------------+-----+
|              5.1|             3.5|              1.4|             0.2|    0|
|              4.9|             3.0|              1.4|             0.2|    0|
|              4.7|             3.2|              1.3|             0.2|    0|
|              4.6|             3.1|              1.5|             0.2|    0|
|              5.0|             3.6|              1.4|             0.2|    0|
+-----------------+----------------+-----------------+----------------+-----+
only showing top 5 rows


In [ ]:
import pandas as pd
from sklearn.datasets import load_iris
iris = load_iris()

df = pd.DataFrame(data=iris.data, columns=iris.feature_names)
df['label'] = iris.target
spark_df = spark.createDataFrame(df)
spark_df.show(5)

+-----------------+----------------+-----------------+----------------+-----+
|sepal length (cm)|sepal width (cm)|petal length (cm)|petal width (cm)|label|
+-----------------+----------------+-----------------+----------------+-----+
|              5.1|             3.5|              1.4|             0.2|    0|
|              4.9|             3.0|              1.4|             0.2|    0|
|              4.7|             3.2|              1.3|             0.2|    0|
|              4.6|             3.1|              1.5|             0.2|    0|
|              5.0|             3.6|              1.4|             0.2|    0|
+-----------------+----------------+-----------------+----------------+-----+
only showing top 5 rows


In [ ]:
from pyspark.ml.feature import VectorAssembler
assembler = VectorAssembler(
    inputCols=iris.feature_names,
    outputCol="features"
)
data = assembler.transform(spark_df)
data = data.select("features", "label")
data.show(5)
train_data, test_data = data.randomSplit([0.8, 0.2], seed=42)

+-----------------+-----+
|         features|label|
+-----------------+-----+
|[5.1,3.5,1.4,0.2]|    0|
|[4.9,3.0,1.4,0.2]|    0|
|[4.7,3.2,1.3,0.2]|    0|
|[4.6,3.1,1.5,0.2]|    0|
|[5.0,3.6,1.4,0.2]|    0|
+-----------------+-----+
only showing top 5 rows


In [ ]:
from pyspark.ml.classification import LogisticRegression
lr = LogisticRegression(featuresCol='features', labelCol='label')
model = lr.fit(train_data)
predictions = model.transform(test_data)
predictions.select("features", "label", "prediction").show()

+-----------------+-----+----------+
|         features|label|prediction|
+-----------------+-----+----------+
|[4.4,3.0,1.3,0.2]|    0|       0.0|
|[4.6,3.2,1.4,0.2]|    0|       0.0|
|[4.6,3.6,1.0,0.2]|    0|       0.0|
|[4.8,3.1,1.6,0.2]|    0|       0.0|
|[4.9,3.1,1.5,0.2]|    0|       0.0|
|[5.0,3.2,1.2,0.2]|    0|       0.0|
|[5.0,3.6,1.4,0.2]|    0|       0.0|
|[5.1,3.8,1.5,0.3]|    0|       0.0|
|[5.4,3.7,1.5,0.2]|    0|       0.0|
|[5.4,3.9,1.3,0.4]|    0|       0.0|
|[5.4,3.9,1.7,0.4]|    0|       0.0|
|[5.5,3.5,1.3,0.2]|    0|       0.0|
|[5.6,2.5,3.9,1.1]|    1|       1.0|
|[5.7,3.8,1.7,0.3]|    0|       0.0|
|[6.1,2.8,4.0,1.3]|    1|       1.0|
|[6.4,3.2,4.5,1.5]|    1|       1.0|
|[4.9,2.5,4.5,1.7]|    2|       2.0|
|[5.5,2.4,3.8,1.1]|    1|       1.0|
|[5.5,2.5,4.0,1.3]|    1|       1.0|
|[5.7,2.9,4.2,1.3]|    1|       1.0|
+-----------------+-----+----------+
only showing top 20 rows


In [ ]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)
accuracy = evaluator.evaluate(predictions)
print("Accuracy:", accuracy)

**Build a clustering model with spark with a dataset of your choice (Marks dataset)**

In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("Student Clustering") \
    .getOrCreate()
df = spark.read.csv("Marks Data Set.csv", header=True, inferSchema=True)
df.show(5)

+----+---+---+---+-------+------+------+---+-------+-----+-----+----+---+---+-----+-----+------+-----+-------+----+----+----+----+----+
|SLNO|ENG|PHY|CHE|HISTORY|GEOGRA|SOCIAL|ENV|SCIENCE|DRAWI|CROFT|SUPW| SW|SWG|SUPWG|SCI G|SOCI G|PHY G|CROFT G|_c19|_c20|_c21|_c22|_c23|
+----+---+---+---+-------+------+------+---+-------+-----+-----+----+---+---+-----+-----+------+-----+-------+----+----+----+----+----+
|   1| 14| 35| 14|     30|    23|    32| 13|     30|   17|   64|  22| 42| A+|  A++|   B+|   A++|  A++|      A|NULL|NULL|NULL|NULL|NULL|
|   2| 12| 12| 13|     30|    21|    22| 13|     32|    8|   34|  18| 25| FF|  A++|    C|     O|    C|      C|NULL|NULL|NULL|NULL|NULL|
|   3|  6|  0|  7|      6|     0|     0|  0|     13|    0|    0|   9| 15| FF|   FF|   FF|    FF|   FF|     FF|NULL|NULL|NULL|NULL|NULL|
|   4| 13| 26| 13|     28|    23|    35| 12|     27|   15|   37|  20| 41| B+|  A++|   B+|    A+|    B|      A|NULL|NULL|NULL|NULL|NULL|
|   5| 20| 47| 15|     34|    29|    62| 14|    

In [ ]:
cols_to_drop = [col for col in df.columns if "Unnamed" in col or col == "SLNO"]
df = df.drop(*cols_to_drop)
df.show()
feature_cols = df.columns

+---+---+---+-------+------+------+---+-------+-----+-----+----+---+---+-----+-----+------+-----+-------+----+----+----+----+----+
|ENG|PHY|CHE|HISTORY|GEOGRA|SOCIAL|ENV|SCIENCE|DRAWI|CROFT|SUPW| SW|SWG|SUPWG|SCI G|SOCI G|PHY G|CROFT G|_c19|_c20|_c21|_c22|_c23|
+---+---+---+-------+------+------+---+-------+-----+-----+----+---+---+-----+-----+------+-----+-------+----+----+----+----+----+
| 14| 35| 14|     30|    23|    32| 13|     30|   17|   64|  22| 42| A+|  A++|   B+|   A++|  A++|      A|NULL|NULL|NULL|NULL|NULL|
| 12| 12| 13|     30|    21|    22| 13|     32|    8|   34|  18| 25| FF|  A++|    C|     O|    C|      C|NULL|NULL|NULL|NULL|NULL|
|  6|  0|  7|      6|     0|     0|  0|     13|    0|    0|   9| 15| FF|   FF|   FF|    FF|   FF|     FF|NULL|NULL|NULL|NULL|NULL|
| 13| 26| 13|     28|    23|    35| 12|     27|   15|   37|  20| 41| B+|  A++|   B+|    A+|    B|      A|NULL|NULL|NULL|NULL|NULL|
| 20| 47| 15|     34|    29|    62| 14|     33|   29|   65|  29| 64|  O|    O|    O

In [ ]:
from pyspark.ml.feature import VectorAssembler
numeric_feature_cols = [col for col in feature_cols if df.schema[col].dataType.typeName() != 'string']
assembler = VectorAssembler(
    inputCols=numeric_feature_cols,
    outputCol="features"
)
data = assembler.transform(df)
data = data.select("features")

data.show(5)

+--------------------+
|            features|
+--------------------+
|[14.0,35.0,14.0,3...|
|[12.0,12.0,13.0,3...|
|(12,[0,2,3,7,10,1...|
|[13.0,26.0,13.0,2...|
|[20.0,47.0,15.0,3...|
+--------------------+
only showing top 5 rows


In [ ]:
from pyspark.ml.clustering import KMeans
kmeans = KMeans(k=3, seed=42)   # 3 clusters (low, medium, high)
model = kmeans.fit(data)
predictions = model.transform(data)
predictions.show()

+--------------------+----------+
|            features|prediction|
+--------------------+----------+
|[14.0,35.0,14.0,3...|         1|
|[12.0,12.0,13.0,3...|         2|
|(12,[0,2,3,7,10,1...|         0|
|[13.0,26.0,13.0,2...|         2|
|[20.0,47.0,15.0,3...|         1|
|[15.0,35.0,14.0,3...|         1|
|[15.0,29.0,14.0,3...|         1|
|[13.0,31.0,14.0,3...|         2|
|[5.0,16.0,11.0,21...|         0|
|[13.0,23.0,14.0,3...|         2|
|[15.0,30.0,14.0,3...|         1|
|[12.0,26.0,14.0,2...|         2|
|[18.0,41.0,14.0,3...|         1|
|[12.0,22.0,14.0,2...|         2|
|[11.0,17.0,14.0,2...|         2|
|[19.0,37.0,15.0,3...|         1|
|[17.0,39.0,14.0,3...|         1|
|[12.0,36.0,12.0,2...|         1|
|[12.0,22.0,12.0,2...|         2|
|[17.0,47.0,15.0,3...|         1|
+--------------------+----------+
only showing top 20 rows


In [ ]:
from pyspark.ml.evaluation import ClusteringEvaluator
evaluator = ClusteringEvaluator()
silhouette = evaluator.evaluate(predictions)
print("Silhouette Score:", silhouette)
centers = model.clusterCenters()
for i, center in enumerate(centers):
    print(f"Cluster {i}: {center}")

Silhouette Score: 0.579925611728548
Cluster 0: [ 7.78106509 12.79289941  9.0295858  14.68639053 11.50887574 11.19526627
 10.08284024 27.15976331  4.00591716 15.19526627 14.66863905 18.0295858 ]
Cluster 1: [14.70418848 35.43979058 13.35078534 29.69109948 25.29319372 45.98167539
 13.51308901 31.30628272 19.22251309 53.9973822  24.43193717 50.60209424]
Cluster 2: [11.78173719 22.85523385 12.14699332 25.74164811 19.922049   34.20489978
 12.6325167  30.28730512 11.45657016 38.08908686 18.4454343  32.85300668]


**Build a Recommendation Engine with Spark with a dataset of your choice (Movies dataset)**

In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("User Clustering MovieLens") \
    .getOrCreate()

In [ ]:
ratings = spark.read.csv(
    "ratings.csv",
    header=True,
    inferSchema=True
)
ratings.show(5)

+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
|     1|      1|   4.0|964982703|
|     1|      3|   4.0|964981247|
|     1|      6|   4.0|964982224|
|     1|     47|   5.0|964983815|
|     1|     50|   5.0|964982931|
+------+-------+------+---------+
only showing top 5 rows


In [ ]:
from pyspark.sql.functions import avg, count
user_features = ratings.groupBy("userId").agg(
    avg("rating").alias("avg_rating"),
    count("movieId").alias("num_movies")
)
user_features.show(5)
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=["avg_rating", "num_movies"],
    outputCol="features"
)

data = assembler.transform(user_features)
data = data.select("features")

data.show(5)

+------+------------------+----------+
|userId|        avg_rating|num_movies|
+------+------------------+----------+
|   148|3.7395833333333335|        48|
|   463| 3.787878787878788|        33|
|   471|             3.875|        28|
|   496| 3.413793103448276|        29|
|   243| 4.138888888888889|        36|
+------+------------------+----------+
only showing top 5 rows
+--------------------+
|            features|
+--------------------+
|[3.73958333333333...|
|[3.78787878787878...|
|        [3.875,28.0]|
|[3.41379310344827...|
|[4.13888888888888...|
+--------------------+
only showing top 5 rows


In [ ]:
from pyspark.ml.feature import StandardScaler
scaler = StandardScaler(inputCol="features", outputCol="scaled_features")
scaler_model = scaler.fit(data)
scaled_data = scaler_model.transform(data)
data = scaled_data.select("scaled_features") \
                  .withColumnRenamed("scaled_features", "features")

In [ ]:
from pyspark.ml.clustering import KMeans
kmeans = KMeans(k=3, seed=42)
model = kmeans.fit(data)
predictions = model.transform(data)
predictions.show()

+--------------------+----------+
|            features|prediction|
+--------------------+----------+
|[7.78050051350494...|         2|
|[7.88098304735876...|         2|
|[8.06224565744802...|         2|
|[7.10266803192306...|         1|
|[8.61128747641401...|         2|
|[6.65785447840868...|         1|
|[8.32231809801085...|         2|
|[8.15587173605064...|         2|
|[7.68213978277925...|         2|
|[7.71038294374535...|         2|
|[8.27805044855335...|         2|
|[10.1315176845349...|         2|
|[7.89396349002500...|         2|
|[7.34406396378366...|         1|
|[8.38351161343741...|         2|
|[8.63969463564686...|         2|
|[10.4028976225135...|         2|
|[5.34330650610924...|         1|
|[5.83904576231407...|         1|
|[6.76188345463382...|         1|
+--------------------+----------+
only showing top 20 rows


In [ ]:
from pyspark.ml.evaluation import ClusteringEvaluator
evaluator = ClusteringEvaluator()
silhouette = evaluator.evaluate(predictions)
print("Silhouette Score:", silhouette)
centers = model.clusterCenters()
for i, center in enumerate(centers):
    print(f"Cluster {i}: {center}")


Silhouette Score: 0.6152652046456408
Cluster 0: [6.88882844 3.80488583]
Cluster 1: [6.71832384 0.46769141]
Cluster 2: [8.30835746 0.38512616]
